In [1]:
!pip install requests beautifulsoup4 pandas openpyxl

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Configuration & Multi-URL Setup
# Updated CITIES dictionary with few specific cities PLUS the general all_india page
CITIES = {
    # General Page Added
    "all_india": "https://droom.in/cars/used",
    # Existing Cities
    "hyderabad": "https://droom.in/cars/used/hyderabad",
    "delhi": "https://droom.in/cars/used/delhi",
    # Other Cities
    "mumbai": "https://droom.in/cars/used/mumbai",
    "pune": "https://droom.in/cars/used/pune",
    "bangalore": "https://droom.in/cars/used/bangalore",
    "chennai": "https://droom.in/cars/used/chennai",
    "gurgaon": "https://droom.in/cars/used/gurgaon",
    "jaipur": "https://droom.in/cars/used/jaipur",
    "kolkata": "https://droom.in/cars/used/kolkata",
}
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
}


# 2. Function to scrape data
def scrape_droom_cars(url, city_name, headers):
    """Fetches and parses a single Droom search page."""
    
    # Error handling and initial fetch 
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {city_name.upper()} page: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    cars_data = []

    # Find all Car Name elements
    car_name_elements = soup.find_all('h5', class_='w-[89%] text-base font-medium')
    
    for name_element in car_name_elements:
        car = {}
        
        #  Car Name 
        car['Name'] = name_element.text.strip()
        
        #  CRITICAL FIX: Find Parent Container for Scoping 
        # Traverse up 3 levels to find a common, large container for the listing
        listing_container = name_element.find_parent('div').find_parent('div').find_parent('div') 
        
        if not listing_container:
            continue
            
        # --- Price ---
        price_element = listing_container.find('h6', class_='font-semibold text-[#30343e]')
        car['Price'] = price_element.text.strip().replace('₹', '').replace(',', '') if price_element else 'N/A'
        
        # --- Other Details (KMs, Location, Fuel, Gear) ---
        detail_spans = listing_container.find_all('span', class_='ps-2 text-xs font-thin capitalize text-[#30343e]')
        
        details = [span.text.strip() for span in detail_spans]
        
        # Assign values based on the expected order
        car['KMs'] = details[0] if len(details) > 0 else 'N/A'
        # Set the Location field to the current city name (from URL) for clarity
        car['Location'] = city_name.replace('_', ' ').title() # Title case for readability
        car['Fuel Type'] = details[2] if len(details) > 2 else 'N/A'
        car['Transmission'] = details[3] if len(details) > 3 else 'N/A'
        
        cars_data.append(car)

    return cars_data


# 3. Execution and Saving
if __name__ == "__main__":
    
    all_car_listings = []
    
    # Loop through all configured cities
    for city, url in CITIES.items():
        print(f"\n--- SCRAPING: {city.replace('_', ' ').upper()} ---")
        
        # Scrape data for the current city
        car_listings = scrape_droom_cars(url, city, HEADERS)
        
        if car_listings:
            print(f"Successfully scraped **{len(car_listings)}** listings for {city}.")
            all_car_listings.extend(car_listings)
        else:
            print(f"Failed to scrape any data for {city}. This often means the site is blocking access or the URL is invalid.")

    # --- Save Combined Data ---
    if all_car_listings:
        print(f"Total combined listings scraped across all cities/pages: **{len(all_car_listings)}**")
        
        df = pd.DataFrame(all_car_listings)
        output_filename = 'droom cars1.xlsx'
        
        try:
            df.to_excel(output_filename, index=False, engine='openpyxl') 
            print(f"Data successfully saved to **{output_filename}** in XLSX format. ✅")
        except Exception as e:
            print(f"Error saving to Excel file: {e}")
    else:
        print("\nScraping finished, but no data was collected from any city. ❌")


--- SCRAPING: ALL INDIA ---
Successfully scraped **24** listings for all_india.

--- SCRAPING: HYDERABAD ---
Successfully scraped **24** listings for hyderabad.

--- SCRAPING: DELHI ---
Successfully scraped **24** listings for delhi.

--- SCRAPING: MUMBAI ---
Successfully scraped **24** listings for mumbai.

--- SCRAPING: PUNE ---
Successfully scraped **24** listings for pune.

--- SCRAPING: BANGALORE ---
Successfully scraped **24** listings for bangalore.

--- SCRAPING: CHENNAI ---
Successfully scraped **24** listings for chennai.

--- SCRAPING: GURGAON ---
Successfully scraped **24** listings for gurgaon.

--- SCRAPING: JAIPUR ---
Successfully scraped **24** listings for jaipur.

--- SCRAPING: KOLKATA ---
Successfully scraped **24** listings for kolkata.
Total combined listings scraped across all cities/pages: **240**
Data successfully saved to **droom cars1.xlsx** in XLSX format. ✅


In [3]:
import pandas as pd

# Read the Excel file
df = pd.read_excel("droom cars1.xlsx")

# Extract year from 'Name' column
df['Year'] = df['Name'].str.extract(r'(\b\d{4}\b)')  # extract 4-digit year

# Remove the year from the 'Name' column text if you want
df['Name'] = df['Name'].str.replace(r'\b\d{4}\b', '', regex=True).str.strip()

# Save the updated DataFrame as .xlsx file
df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ File saved successfully as 'droom_cars_final.xlsx'")

✅ File saved successfully as 'droom_cars_final.xlsx'


In [5]:
import pandas as pd

df = pd.read_excel("droom_cars_final.xlsx")

df['Brand'] = df['Name'].str.split().str[0]

df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ Brand column added and 'droom_cars_final.xlsx' updated")

✅ Brand column added and 'droom_cars_final.xlsx' updated


In [7]:
import pandas as pd

# Read your Excel file
df = pd.read_excel("droom_cars_final.xlsx")

# Clean 'Price' column — remove non-numeric characters and convert to numbers
df['Price'] = pd.to_numeric(
    df['Price'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Convert price to lakhs (2 decimal points)
df['Price_in_Lakhs'] = (df['Price'] / 100000).round(2).astype(str) + ' Lakhs'

#: Save updated file (same final file)
df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ 'Price_in_Lakhs' column added successfully and saved in 'droom_cars_final.xlsx'")

✅ 'Price_in_Lakhs' column added successfully and saved in 'droom_cars_final.xlsx'


In [9]:
import pandas as pd

# Read your Excel file
df = pd.read_excel("droom_cars_final.xlsx")

# Clean 'Price' column — remove non-numeric characters and convert to numeric
df['Price'] = pd.to_numeric(
    df['Price'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Convert price to lakhs with ₹ symbol and two decimal points
df['Price_in_Lakhs'] = '₹' + (df['Price'] / 100000).round(2).astype(str) + ' Lakhs'

# Save updated file
df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ 'Price_in_Lakhs' column added with ₹ symbol and saved in 'droom_cars_final.xlsx'")

✅ 'Price_in_Lakhs' column added with ₹ symbol and saved in 'droom_cars_final.xlsx'


In [13]:
import pandas as pd

# Read your Excel file
df = pd.read_excel("droom_cars_final.xlsx")

# Clean 'Price' column — remove non-numeric characters and convert to numeric
df['Price'] = pd.to_numeric(
    df['Price'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Create formatted Price column (₹ and Lakhs)
df['Price_in_Lakhs'] = '₹' + (df['Price'] / 100000).round(2).map('{:.2f}'.format) + ' Lakhs'

# Drop old Price column
df.drop(columns=['Price'], inplace=True)

# Rename 'Price_in_Lakhs' → 'Price'
df.rename(columns={'Price_in_Lakhs': 'Price'}, inplace=True)

# Save the updated file
df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ Old 'Price' removed, formatted 'Price' column added, and file saved in 'droom_cars_final.xlsx'")

✅ Old 'Price' removed, formatted 'Price' column added, and file saved in 'droom_cars_final1.xlsx'


In [11]:
import pandas as pd

# Read your Excel file
df = pd.read_excel("droom_cars_final.xlsx")

# Rearrange columns in desired order
new_order = ['Brand', 'Name', 'Location', 'Fuel Type', 'Transmission', 'Year', 'KMs', 'Price']
df = df[new_order]

# Save the updated file
df.to_excel("droom_cars_final.xlsx", index=False)

print("✅ Columns rearranged and file saved in 'droom_cars_final.xlsx'")

✅ Columns rearranged and file saved in 'droom_cars_final.xlsx'
